# Per-taxon counts used for the taxonomic trees

**Produces:** the input tables for Figure 1C and Supplementary Figure 4: number of MS/MS scans (`BA_scan_counts_per_taxonomy_no_artifacts.tsv`) and delta-mass counts (`BA_delta_counts_per_taxonomy_no_artifacts.tsv`) per NCBI taxon and bile acid core.

Only entries with an NCBI taxonomy, positive delta masses (or no delta, for C28 bile alcohols) and not flagged as alternative ion forms are counted.

**Input:** `merged_filepaths_redu_metadata.tsv` (written by the body part notebook) and the ion-form annotation tables.

The trees themselves were drawn in R (taxize, ape); those scripts are added separately to this folder.

In [1]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

# Removing other ion forms

In [2]:
import pandas as pd 

In [3]:
#import table merged with ReDU
df = pd.read_csv(f'{PROJECT_DIR}/body_part_distribution/merged_filepaths_redu_metadata.tsv', sep='\t')

#keep only the rows in which we have an organism
df = df[df['NCBITaxonomy'].isna() == False]
df = df[df['NCBITaxonomy'] != 'missing value']

#keep only the columns we need
df = df[['core_std', 'identifier', 'original_path', 'precmz', 'delta_mass', 
         'delta_round', 'filepath',  'NCBITaxonomy']]

#standardize the core_std names
df['core_std'] = df['core_std'].replace({
    'C24_trihydroxy_part1': 'C24_trihydroxy',
    'C24_trihydroxy_part2': 'C24_trihydroxy',
    'C24_trihydroxy_part3': 'C24_trihydroxy',
    'C24_alkylamine_trihydroxy_part1': 'C24_alkamine_trihydroxy',
    'C24_alkylamine_trihydroxy_part2': 'C24_alkamine_trihydroxy',
    'C24_alkylamine_trihydroxy_part3': 'C24_alkamine_trihydroxy',
    'C24_alkylamine_trihydroxy_part4': 'C24_alkamine_trihydroxy',
    'Aceto_dihydroxy': 'C26_dihydroxy'
})

#consider only positive deltas or missing (for the C28s)
df = df[(df['delta_round'] > 0) | (df['delta_round'].isna())]

In [4]:
#import all the artifact files
c23_artifacts = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C23_combined_annotated_artifacts.tsv', sep='\t').drop_duplicates()
c24_artifacts = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C24_combined_annotated_artifacts.tsv', sep='\t').drop_duplicates()
c24_alkamine_artifacts = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C24_alkylamine_combined_annotated_artifacts.tsv', sep='\t').drop_duplicates()
c26_artifacts = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/Aceto_combined_annotated_artifacts.tsv', sep='\t').drop_duplicates()
c27_artifacts = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C27_combined_annotated_artifacts.tsv', sep='\t').drop_duplicates()
c28_artifacts = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C28_alcohol_combined_annotated_artifacts.tsv', sep='\t').drop_duplicates()

#combine all of them vertically
artifacts_combined = pd.concat([c23_artifacts, c24_artifacts, c24_alkamine_artifacts, c26_artifacts, c27_artifacts, c28_artifacts], axis=0)

merged = pd.merge(df, artifacts_combined[['identifier','is_artifact']], how='left', on=['identifier'])

#keep only the ones that are False (not artifacts)
merged = merged[merged['is_artifact'] == False]

#drop duplicates
merged = merged.drop_duplicates()

/tmp/ipykernel_337866/319316941.py:4: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  c24_alkamine_artifacts = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total/C24_alkylamine_combined_annotated_artifacts.tsv', sep='\t').drop_duplicates()


In [5]:
counts_BA_class = merged.groupby(['NCBITaxonomy', 'core_std']).size().reset_index(name='scan_counts')
counts_BA_deltas = (
    merged.groupby(["NCBITaxonomy", "core_std"])["delta_round"]
      .count()
      .reset_index(name="delta_counts")
)

counts_BA_deltas.to_csv(f'{PROJECT_DIR}/body_part_distribution/BA_delta_counts_per_taxonomy_no_artifacts.tsv', sep='\t', index=False)
counts_BA_class.to_csv(f'{PROJECT_DIR}/body_part_distribution/BA_scan_counts_per_taxonomy_no_artifacts.tsv', sep='\t', index=False)